Dataset: Shell Commands Used by Participants of Hands-on Cybersecurity Training
(https://zenodo.org/records/6670113)

In [1]:
import requests, zipfile, io, os
from st_router.paths import get_rootdir
import pandas as pd
ROOT_DIR = get_rootdir()
DATA_RAW = ROOT_DIR / "data" / "raw"
MASARYK_DIR = DATA_RAW / "masaryk"

In [2]:
r = requests.get("https://zenodo.org/records/6670113/files/data.zip?download=1")
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall(MASARYK_DIR)

# go to README.md and verify version 21762

In [3]:
for training in MASARYK_DIR.iterdir():
    count = 0
    for _ in training.rglob("*.json"):
        count += 1
    print(training.name + ":", count)

House of cards: 22
Junior hacker adaptive: 62
Junior hacker: 18
Locust 3302: 134
README.md: 0
Secret laboratory: 13
SQL injection: 8
Webmin exploit practice: 10


In [4]:
dfs = []

for file in MASARYK_DIR.rglob("*.json"):
    path_parts = file.relative_to(MASARYK_DIR).parts
    training = path_parts[0]
    # only add run if path resolves to 3 or more dirs e.g. house of cards/2019... [0][1] and [2] is json
    if (len(path_parts) >= 3):
        run = path_parts[1]
    else:
        run = None
    df = pd.read_json(file, lines=True)
    df["training"] = training
    df["full_path"] = str(file.relative_to(MASARYK_DIR))
    df["run"] = run
    dfs.append(df)

long_df = pd.concat(dfs, ignore_index=True)

long_df.head()

,hostname,ip,timestamp_str,sandbox_id,cmd,pool_id,wd,cmd_type,username,training,full_path,run,tags
0,attacker,10.1.135.83,2021-06-30 06:25:28.440000+00:00,123,nmap 10.1.26.9,30,/home/kali,bash-command,kali,Secret laboratory,Secret laboratory/sandbox-123-useractions.json,None,NaN
1,attacker,10.1.135.83,2021-06-30 06:30:02.296000+00:00,123,nmap 10.1.26.9 -sV,30,/home/kali,bash-command,kali,Secret laboratory,Secret laboratory/sandbox-123-useractions.json,None,NaN
2,attacker,10.1.135.83,2021-06-30 06:38:17.414000+00:00,123,nmap 10.1.26.9 -sV -p 10000,30,/home/kali,bash-command,kali,Secret laboratory,Secret laboratory/sandbox-123-useractions.json,None,NaN
3,attacker,10.1.135.83,2021-06-30 06:43:01.383000+00:00,123,wget https://raw.githubusercontent.com/foxsin3...,30,/home/kali,bash-command,kali,Secret laboratory,Secret laboratory/sandbox-123-useractions.json,None,NaN
4,attacker,10.1.135.83,2021-06-30 06:43:11.057000+00:00,123,python3 webmin-1.890_exploit.py,30,/home/kali,bash-command,kali,Secret laboratory,Secret laboratory/sandbox-123-useractions.json,None,NaN


In [5]:
# check how many run folders
s = long_df.groupby("training")["run"].nunique()
print(s[s>1])
print()
# check if pool_id can stand in for run
s = long_df.groupby(["training", "run"])["pool_id"].nunique()
print(s[s>1])
print()
# does each pool belong to exactly one run
s = long_df.groupby(["training", "pool_id"])["run"].nunique()
print(s[s>1])
print()

# check if training, pool id, sandbox id can form unique key for user
s = long_df.groupby(["training", "pool_id", "sandbox_id"])["run"].nunique()
print(s)
print()
# they do

# check if only one sandbox id per file
s = long_df.groupby("full_path")["sandbox_id"].nunique()
print(s)
print()
print((s > 1).sum())


training
House of cards            3
Junior hacker adaptive    8
Locust 3302               9
Name: run, dtype: int64

training     run       
Locust 3302  2022-05-04    4
Name: pool_id, dtype: int64

training        pool_id
House of cards  0          3
Locust 3302     1          3
Name: run, dtype: int64

training                 pool_id  sandbox_id
House of cards           0        93            1
                                  95            1
                                  101           1
                                  112           1
                                  113           1
                                               ..
Webmin exploit practice  1        329           0
                                  331           0
                                  351           0
                                  358           0
                                  374           0
Name: run, Length: 267, dtype: int64

full_path
House of cards/2019-08-15 KYPO Summer School/sandb

We can use training, pool id and sandbox id as a unique key for each user.

In [6]:
# joining three unique key columns as string 
long_df["unique_key"] = long_df["training"] + "_" + long_df["pool_id"].astype(str) + "_" + long_df["sandbox_id"].astype(str)

long_df.head()

,hostname,ip,timestamp_str,sandbox_id,cmd,pool_id,wd,cmd_type,username,training,full_path,run,tags,unique_key
0,attacker,10.1.135.83,2021-06-30 06:25:28.440000+00:00,123,nmap 10.1.26.9,30,/home/kali,bash-command,kali,Secret laboratory,Secret laboratory/sandbox-123-useractions.json,None,NaN,Secret laboratory_30_123
1,attacker,10.1.135.83,2021-06-30 06:30:02.296000+00:00,123,nmap 10.1.26.9 -sV,30,/home/kali,bash-command,kali,Secret laboratory,Secret laboratory/sandbox-123-useractions.json,None,NaN,Secret laboratory_30_123
2,attacker,10.1.135.83,2021-06-30 06:38:17.414000+00:00,123,nmap 10.1.26.9 -sV -p 10000,30,/home/kali,bash-command,kali,Secret laboratory,Secret laboratory/sandbox-123-useractions.json,None,NaN,Secret laboratory_30_123
3,attacker,10.1.135.83,2021-06-30 06:43:01.383000+00:00,123,wget https://raw.githubusercontent.com/foxsin3...,30,/home/kali,bash-command,kali,Secret laboratory,Secret laboratory/sandbox-123-useractions.json,None,NaN,Secret laboratory_30_123
4,attacker,10.1.135.83,2021-06-30 06:43:11.057000+00:00,123,python3 webmin-1.890_exploit.py,30,/home/kali,bash-command,kali,Secret laboratory,Secret laboratory/sandbox-123-useractions.json,None,NaN,Secret laboratory_30_123


We now need to gather the 10 consecutive commands of each user in a rolling window. The window of 10 was chosen to reflect the context rules of smart terminal

In [7]:
long_df["timestamp"] = pd.to_datetime(long_df["timestamp_str"], utc=True)
long_df = long_df.drop(columns = "timestamp_str")

long_df = long_df.sort_values(["unique_key", "timestamp"], kind="mergesort")

In [8]:
# TODO: make util helper for col rearranging
first = ["unique_key", "timestamp"]
col_list = first + [c for c in long_df.columns if c not in first]
long_df = long_df[col_list]
long_df.head()

,unique_key,timestamp,hostname,ip,sandbox_id,cmd,pool_id,wd,cmd_type,username,training,full_path,run,tags
20729,House of cards_0_101,2019-09-26 09:23:29.156399+00:00,attacker,10.0.135.83,101,nmap -sV,0,/root,bash-command,root,House of cards,House of cards/2019-09-19 PV276 First Lesson/s...,2019-09-19 PV276 First Lesson,NaN
20730,House of cards_0_101,2019-09-26 09:23:53.451916+00:00,attacker,10.0.135.83,101,nmap --help,0,/root,bash-command,root,House of cards,House of cards/2019-09-19 PV276 First Lesson/s...,2019-09-19 PV276 First Lesson,NaN
20731,House of cards_0_101,2019-09-26 09:24:40.283127+00:00,attacker,10.0.135.83,101,nmap -sV 172.18.1.5,0,/root,bash-command,root,House of cards,House of cards/2019-09-19 PV276 First Lesson/s...,2019-09-19 PV276 First Lesson,NaN
20732,House of cards_0_101,2019-09-26 09:31:56.239241+00:00,attacker,10.0.135.83,101,nmap --help,0,/root,bash-command,root,House of cards,House of cards/2019-09-19 PV276 First Lesson/s...,2019-09-19 PV276 First Lesson,NaN
20733,House of cards_0_101,2019-09-26 09:36:58.199192+00:00,attacker,10.0.135.83,101,search libssh,0,NaN,msf-command,NaN,House of cards,House of cards/2019-09-19 PV276 First Lesson/s...,2019-09-19 PV276 First Lesson,NaN


In [9]:
# all cols with non-empty tags are the only ones with empty cmds -- drop
print(long_df["tags"].notnull().sum())
print(long_df["cmd"].isna().sum())

long_df = long_df.dropna(subset=["cmd"])

# dropping useless cols
long_df = long_df.drop(columns=["username", "ip", "full_path", "pool_id", "sandbox_id", "run", "tags"])

19
19


Need to decide when a session ends. Can group by timestamp differences and decide on threshold between consecutive commands which constitutes a session break.

In [10]:
long_df = long_df.reset_index(drop=True)
long_df["gap_s"] = long_df.groupby("unique_key")["timestamp"].diff().dt.total_seconds()

In [11]:
display(long_df.describe())
display(long_df["gap_s"].quantile([0.9, 0.95, 0.99, 0.999]))

,gap_s
count,21125.000000
mean,294.363824
std,8909.016306
min,0.000000
25%,3.599097
50%,12.867000
75%,46.711000
max,721217.754000


0.900      132.821800
0.950      239.513600
0.990      872.701120
0.999    36101.033815
Name: gap_s, dtype: float64